In [ ]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from itertools import combinations
from itertools import product

from sklearn.utils.parallel import Parallel, delayed

%run Model_functions.ipynb

In [ ]:
BASE_DIR = "../../DATA/AGB_DATA/Merged_Data/"
#BASE_DIR = "../../Data/"

# SENTINEL + CANOPY DATA

In [ ]:
SENTINEL_DATA_CSV        = BASE_DIR + "/Sentinel_AGB/AGB_SENTINEL_CANOPY.csv"
sentinel_raw_df = pd.read_csv(SENTINEL_DATA_CSV)
print(sentinel_raw_df.shape)
assert len(sentinel_raw_df["simard_height_m"].head())
assert len(sentinel_raw_df["tandemx_height_m"].head())

In [ ]:
sentinel_raw_df.columns

## Sentinel feature selection

In [ ]:
sent_non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    'dataset',             # metadata
    'sentinel_time',       # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
    'diameter',            # Allometric
    'height',               # Allometric
    'cloud_threshold_used',
    'start_date'
]

SENT_interact_cols = [
    # Indices (excluding MNDWI which isn't a biomass index)
    'EVI', 'NBR', 'NDVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge',
    # Raw structural bands
    'NIR', 'SWIR1', 'SWIR2',
]

target = 'plant_AGB_kg'

sent_feature_cols = [c for c in sentinel_raw_df.columns if c not in sent_non_feature_cols]

### Feature selection with Simard canopy height

In [ ]:
X_sent_t = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][sent_feature_cols].copy()

# Select TANDEMX
X_sent_t = X_sent_t.rename({'tandemx_height_m': 'height'}, axis=1)
X_sent_t = X_sent_t.drop(columns=['simard_height_m'])

In [ ]:
X_sent_t.shape

In [ ]:
X_sent_t.columns

### Create interaction terms

In [ ]:
X_sent_t = create_interact_terms(X_sent_t, SENT_interact_cols)

### Feature selection with Simard canopy height

In [ ]:
X_sent_s = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][sent_feature_cols].copy()

# Select SIMARD
X_sent_s = X_sent_s.rename({'simard_height_m': 'height'}, axis=1)
X_sent_s = X_sent_s.drop(columns=['tandemx_height_m'])

### Create interaction terms

In [ ]:
X_sent_s = create_interact_terms(X_sent_s, SENT_interact_cols)

### Create target variable

In [ ]:
y_sent = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][target].copy()

## Sentinel test features

In [ ]:
test_cv = 5

In [ ]:
sent_struct_features = ['height']
species              = ['species']

sent_bands   = ['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2']
sent_indices = ['NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']

sent_top_spectral  = ['EVI', 'NIR', 'NBR', 'SWIR1']
sent_redband       = ['RE1', 'RE2', 'RE3']
sent_interaction_terms = [c for c in X_sent_t.columns if c.startswith('height_x_')]

# Curated interactions matching curated spectral sets
sent_top_spectral_interactions = [f'height_x_{b}' for b in sent_top_spectral
                                  if f'height_x_{b}' in sentinel_raw_df.columns]
sent_redband_interactions      = [f'height_x_{b}' for b in sent_redband
                                  if f'height_x_{b}' in sentinel_raw_df.columns]

sent_features_list = [
    # Non-spectral baselines
    #struct_features,  # already tested in EMIT
    #species,          # already tested in EMIT
    sent_struct_features + species,

    # Spectral alone
    sent_bands,
    sent_indices,
    sent_top_spectral,
    sent_redband,

    # Spectral + height
    sent_bands        + sent_struct_features,
    sent_indices      + sent_struct_features,
    sent_top_spectral + sent_struct_features,
    sent_redband      + sent_struct_features,

    # Spectral + height + species
    sent_bands        + sent_struct_features + species,
    sent_top_spectral + sent_struct_features + species,

    # Spectral + height + interactions (curated where it matters)
    sent_bands        + sent_struct_features + sent_interaction_terms,
    sent_top_spectral + sent_struct_features + sent_top_spectral_interactions,
    sent_redband      + sent_struct_features + sent_redband_interactions,

    # Full specturm of features
    sent_bands        + sent_indices + sent_struct_features + sent_interaction_terms + species
]

## Sentiel. Create plot groups.

In [ ]:
# Retain the groups/plot_id for splitting the data based on groups.
plot_groups_sent = None
if 'plot_id' in X_sent_t.columns:
    plot_groups_sent = X_sent_t['plot_id'].copy()
    X_sent_t = X_sent_t.drop(columns=['plot_id'])

if 'plot_id' in X_sent_s.columns:
    X_sent_s = X_sent_s.drop(columns=['plot_id'])

near_zero_sites_sent, high_agb_sites_sent,\
    near_zero_plots_sent, high_agb_plots_sent = \
        get_low_and_high_agb_plots(y_sent,
                                   plot_groups_sent)

grp_sentinel = GROUP_INFO(near_zero_sites_sent,
                          high_agb_sites_sent,
                          near_zero_plots_sent,
                          high_agb_plots_sent,
                          groups=plot_groups_sent,
                          cv=test_cv)

In [ ]:
assert plot_groups_sent is not None

# SENTINEL-2 EXPERIMENTS

In [ ]:
test_cv = 5

In [ ]:
LINEAR_FULL    = ["linear", "ridge", "lasso", "elasticnet"]
LINEAR_NO_OLS  = ["ridge", "lasso", "elasticnet"]

data_combos= [("SENTINEL + TANDEMX", X_sent_t, y_sent, sent_features_list, grp_sentinel, LINEAR_FULL),
              ("SENTINEL + SIMARD" , X_sent_s, y_sent, sent_features_list, grp_sentinel, LINEAR_FULL)
             ]
model_list = ["linear", "rf", "xgboost", "lgbm", "merf"]
is_grids   = [False, True]
is_groups  = [True]

In [ ]:
global_experiment_list = {}
experiments_by_category = {}

for (combo_name, X, y, features_list, grp, linear_variants), model, use_grid, use_groups in product(
    data_combos, model_list, is_grids, is_groups
):
    print(f"\n{'='*70}")
    print(f"DATA: {combo_name} | MODEL: {model} | GRID: {use_grid} | GROUPED: {use_groups}")
    print('='*70)

    #run_label = f"{combo_name}, {model}, grid_{'yes' if use_grid else 'no'}, GROUPED: {'yes' if use_groups else 'no'}"

    exp_result = run_experiment(
        X, y,
        is_groups       = use_groups,
        group_info      = grp,
        features_list   = features_list,
        model_type      = model,
        linear_variants = linear_variants,
        is_grid         = use_grid,
        is_stratified   = False,
        exp_total_list = global_experiment_list,
        exp_by_category=experiments_by_category,
        label           = combo_name
    )

# FIND THE BEST EXPERIMENT SO FAR.

## SUMMARY OF EXPERIMENTS

In [ ]:
%run Model_functions.ipynb
best_results = filter_best_experiments(global_experiment_list)
tab_df = tabulate_results(best_results)

## BEST ONE

In [ ]:
%run Model_functions.ipynb
best_result = best_results[0][1]
print_experiment(best_result)